# Exercise — Module 7: Anomaly Detection

**Dataset (โปรดอ่านก่อนใช้งาน):**
- *NAB `realKnownCause/machine_temperature_system_failure`* — Ahmad, S., Lavin, A., Purdy, S., Agha, Z. (2017). "Unsupervised real-time anomaly detection for streaming data", *Neurocomputing*, 262, 134–147. Numenta Anomaly Benchmark, https://github.com/numenta/NAB (MIT License) — `datasets/machine_temperature_system_failure.csv`
- *labels* — สำเนาจาก `combined_windows.json` ของ NAB (repo เดียวกัน, MIT License) — `datasets/nab_machine_temperature_labels.json` (4 labeled windows)

งานต่อเนื่องจาก book.ipynb: ปรับ detector ทั้งสามตัวบนซีรีส์เดิม แล้วปิดด้วยคำถามออกแบบระบบจริง ใช้ `random_state=42` เสมอเมื่อมีความสุ่ม

---

**เวลาที่คาดหวังสำหรับแบบฝึกหัดนี้:** ~45 นาที (ทำในห้องช่วง lab — เฉลยเปิดเผยหลังจบช่วง)


In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit(
        "ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run "
        "jupyter lab)"
    )


DATA = data_dir()


## เตรียมข้อมูล (ให้มาแล้ว — รันผ่านก่อนทำข้อ)

ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)


In [2]:
import polars as pl  # noqa: E402
import numpy as np  # noqa: E402
import json  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams["font.family"] = [
    "DejaVu Sans",
    "Noto Sans Thai",
    "Leelawadee UI",
    "Tahoma",
    "Thonburi",
]

df = pl.read_csv(
    DATA / "machine_temperature_system_failure.csv", try_parse_dates=True
)
ts = df["timestamp"].to_numpy()
v = df["value"].to_numpy()

lab_windows = json.loads(
    (DATA / "nab_machine_temperature_labels.json").read_text()
)["realKnownCause/machine_temperature_system_failure.csv"]
lab_mask = np.zeros(df.height, dtype=bool)
for a, b in lab_windows:
    lab_mask |= (ts >= np.datetime64(a)) & (ts <= np.datetime64(b))


def evaluate(flag: np.ndarray, name: str) -> tuple[int, int]:
    """นับ labeled windows ที่ถูกจับ (overlap แบบง่าย) + false alarms "
    "นอกหน้าต่าง."""
    caught = sum(
        1
        for a, b in lab_windows
        if (((ts >= np.datetime64(a)) & (ts <= np.datetime64(b))) & flag).any()
    )
    false_alarms = int((flag & ~lab_mask).sum())
    print(
        f"{name}: จับได้ {caught}/{len(lab_windows)} windows | flags = "
        f"{int(flag.sum())} | false alarms = {false_alarms}"
    )
    return caught, false_alarms

## ข้อ A1 — ปรับ z-threshold: จับให้ครบ 4 หน้าต่าง โดย false alarms ไม่เกิน 1,200

ใน book เราใช้ `|z| > 5` แล้วได้ 3/4 (หน้าต่างที่ 3 — ทางลาดช้า — หลุด)

1. คำนวณ `robust_z` แบบเดียวกับ book (rolling median + rolling MAD, หน้าต่าง trailing `W = 144`)
2. ไล่ threshold ใน `[5.0, 4.0, 3.5, 3.3, 3.0]` รายงาน (caught, false alarms) ของแต่ละค่า
3. เลือก threshold ที่ **จับครบ 4 หน้าต่าง และ false alarms ≤ 1,200** — รายงานค่าที่เลือก
4. ตอบสั้น ๆ: ตอนนี้ detector จับได้ครบแล้ว แลกมาด้วยอะไร?

TODO


In [3]:
W = 144

s = df["value"]
roll_med = s.rolling_median(window_size=W, min_samples=W // 2)
roll_mad = (s - roll_med).abs().rolling_median(
    window_size=W, min_samples=W // 2
) * 1.4826
robust_z = ((s - roll_med) / roll_mad).fill_null(0.0).fill_nan(0.0)

for thr in [5.0, 4.0, 3.5, 3.3, 3.0]:
    flag = ...
    evaluate(flag, f"z thr={thr}")

best_thr = ...
print(f"threshold ที่เลือก = {best_thr}")

TypeError: unsupported operand type(s) for &: 'bool' and 'ellipsis'

**คำใบ้:** flag = `(robust_z.abs() > thr).to_numpy()` — สังเกตว่ายิ่งลด threshold false alarms โตเร็วกว่าที่ caught เพิ่ม — นี่คือเส้นโค้ง trade-off แบบเดียวกับ ROC ที่เราเจอใน Module 5

## ข้อ A2 — เปลี่ยนขนาดหน้าต่างของ Isolation Forest แล้วดูว่า detection ขยับอย่างไร

ใน book ใช้ `WIN = 12` (1 ชั่วโมง) ได้ 2/4, false alarms ~84

1. เขียนฟังก์ชันสร้าง window features (mean, std, slope, min, max — ไม่ overlap) ที่รับ `win` เป็นพารามิเตอร์
2. วน `win` ใน `[6, 12, 24, 48]` fit `IsolationForest(contamination=0.01, random_state=42)` ทุกครั้งใหม่ แล้ว `evaluate`
3. รายงาน: ขนาดหน้าต่างไหนจับได้มากที่สุด / หน้าต่างไหน false alarms น้อยที่สุด และอธิบายว่าทำไมหน้าต่างใหญ่จึงพลาด anomaly แบบสั้นมากขึ้น

TODO

TODO

TODO


In [4]:
from sklearn.ensemble import IsolationForest  # noqa: E402,F401

CONTAM = 0.01


def window_features(values: np.ndarray, win: int) -> np.ndarray:
    """คืน feature matrix (n_windows, 5): mean, std, slope, min, max แบบไม่ "
    "overlap."""
    ...


def detect_if(values: np.ndarray, win: int) -> np.ndarray:
    """คืน boolean flag ต่อ sample (เผยแพร่ flag หน้าต่างกลับไปทุก sample "
    "ในหน้าต่าง)."""
    ...


for win in [6, 12, 24, 48]:
    evaluate(..., f"IF win={win}")

TypeError: unsupported operand type(s) for &: 'bool' and 'ellipsis'

**คำใบ้:** slope ในหน้าต่าง = `np.polyfit(np.arange(win), seg, 1)[0]` — เผยแพร่ flag กลับด้วย `np.repeat(pred, win)` แล้วเขียนลง boolean array ขนาด `len(values)`

## ข้อ B1 — One-Class SVM สองค่า nu

fit บนช่วงปกติยาวกลางซีรีส์ (ระหว่างหน้าต่าง 2 → 3) เหมือน book แต่เทียบ `nu = 0.01` กับ `nu = 0.05` (gamma = 0.05 คงที่):

1. fit สองโมเดล, score ทั้งซีรีส์, `evaluate` ทั้งคู่
2. รายงานว่า nu ที่มากขึ้นเปลี่ยนจำนวน flags/false alarms อย่างไร และเชื่อมกับนิยามของ nu ใน book หัวข้อ 5

TODO


In [5]:
from sklearn.pipeline import make_pipeline  # noqa: E402,F401
from sklearn.preprocessing import StandardScaler  # noqa: E402,F401
from sklearn.svm import OneClassSVM  # noqa: E402,F401

train_mid = (ts > np.datetime64(lab_windows[1][1])) & (
    ts < np.datetime64(lab_windows[2][0])
)
GAMMA = 0.05

for nu in [0.01, 0.05]:
    ocsvm = ...
    oc_flag = ...
    evaluate(oc_flag, f"OCSVM nu={nu}, gamma={GAMMA}")

TypeError: unsupported operand type(s) for &: 'bool' and 'ellipsis'

**คำใบ้:** โครงเดียวกับ book: `make_pipeline(StandardScaler(), OneClassSVM(nu=nu, gamma=GAMMA))` — fit ด้วย `v[train_mid].reshape(-1, 1)` แล้ว predict ด้วย `v.reshape(-1, 1)`; ค่าที่คืนเป็น −1 คือ anomalous

## ข้อ C1 — ลงมือเขียน CUSUM detector เอง แล้ววัดบนซีรีส์จริง

CUSUM จาก book หัวข้อ 6 ทำงานบน residual `r = value − rolling median` โดยรวมสองทิศทาง:

- `pos_sum = max(0, pos_sum + r − K)` (กองเมื่อ residual ล้วไปทางบวก)
- `neg_sum = max(0, neg_sum − r − K)` (ทางลบ)
- alarm เมื่อฝั่งใดทะลุ `H` แล้ว reset ทั้งสองฝั่งเป็นศูนย์

ใช้ `K = 0.5·σ`, `H = 5·σ` เมื่อ `σ = median(rolling MAD)` — แล้ว `evaluate` บนซีรีส์จริง

คำถาม: CUSUM จับหน้าต่างไหนที่ robust z (thr=5) พลาดไป? ทำไมจึงจับได้

TODO


In [6]:
K_MULT = 0.5
H_MULT = 5.0

sigma = float(np.nanmedian(roll_mad.to_numpy()))
resid = v - roll_med.to_numpy()

pos_sum = neg_sum = 0.0
cusum_flag = np.zeros(len(v), dtype=bool)
for i in range(1, len(v)):
    r = resid[i] if np.isfinite(resid[i]) else 0.0
    ...
    if ...:
        cusum_flag[i] = True
        pos_sum = neg_sum = 0.0

evaluate(cusum_flag, f"CUSUM K={K_MULT}·σ, H={H_MULT}·σ")

CUSUM K=0.5·σ, H=5.0·σ: จับได้ 4/4 windows | flags = 22694 | false alarms = 20426


(4, 20426)

**คำใบ้:** ภายในลูป: `pos_sum = max(0.0, pos_sum + r - K_MULT * sigma)` และ `neg_sum = max(0.0, neg_sum - r - K_MULT * sigma)`; เงื่อนไข alarm คือ `pos_sum > H_MULT * sigma or neg_sum > H_MULT * sigma` — ลูปนี้ O(n) รันได้เร็วแม้ 22.7k จุด

## ข้อ D1 — โจทย์คิด (ตอบใน markdown)

หน่วยงานของคุณต้องตั้ง **monitoring dashboard** เฝ้าเครื่องมือสอบเทียบในห้องปฏิบัติการ (ตัวอย่างเช่นเซนเซอร์อุณหภูมิของตู้เชื่อถือ) ให้เลือก detector **หนึ่งตัวหลัก** จากสี่ตัวในโมดูลนี้ (robust z / Isolation Forest / One-Class SVM / CUSUM) พร้อมเหตุผล โดยพิจารณาอย่างน้อย:

- ความเสี่ยงจาก false alarms ต่อความเชื่อมั่นของทีมเฝ้าระบบ (alert fatigue)
- ความสามารถอธิบายให้ผู้ตรวจสอบ/auditor ฟัง (traceability)
- พฤติกรรมที่ห่วงกังวลที่สุดของเครื่องมือสอบเทียบ: sudden, dropout หรือ slow drift

คำตอบข้อ D1: